# Pose Visualization & Coordinate System Verification

This notebook provides tools for visualizing human poses and verifying coordinate system consistency across different datasets.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from pathlib import Path
# Set matplotlib to display in notebook
%matplotlib inline

In [ ]:
# ============================================================
# Coordinate System Verification: RealWorld-1225 vs MMBody
# ============================================================
# This cell compares pose data from different datasets to verify
# coordinate system consistency after applying transforms.

import numpy as np
import glob
import sys
from pathlib import Path

# Get workspace root (3 levels up from this notebook: src/wicompass/evaluation/)
WORKSPACE_ROOT = Path(__file__).resolve().parent.parent.parent.parent if '__file__' in dir() else Path.cwd()
# Fallback: try to find workspace root by looking for known directories
if not (WORKSPACE_ROOT / 'datasets').exists():
    for candidate in [Path.cwd(), Path.cwd().parent, Path("../../..").resolve()]:
        if (candidate / 'datasets').exists():
            WORKSPACE_ROOT = candidate
            break

print(f"Workspace root: {WORKSPACE_ROOT}")

sys.path.insert(0, str(WORKSPACE_ROOT / 'src/wicompass/dataset'))
from dataset import PreprocessingUtils, get_available_datasets, create_dataset

def analyze_pose_orientation(joints, name):
    """Analyze the orientation of a pose based on pelvis-to-head vector."""
    pelvis_to_head = joints[15] - joints[0]
    height_axis = np.argmax(np.abs(pelvis_to_head))
    axis_names = ["X", "Y", "Z"]
    print(f"\n【{name}】")
    print(f"  Pelvis: [{joints[0][0]:.3f}, {joints[0][1]:.3f}, {joints[0][2]:.3f}]")
    print(f"  Head:   [{joints[15][0]:.3f}, {joints[15][1]:.3f}, {joints[15][2]:.3f}]")
    print(f"  Pelvis→Head: [{pelvis_to_head[0]:.3f}, {pelvis_to_head[1]:.3f}, {pelvis_to_head[2]:.3f}]")
    print(f"  Height axis: {axis_names[height_axis]} (value: {pelvis_to_head[height_axis]:.3f})")
    if pelvis_to_head[height_axis] < 0:
        print(f"  ⚠️ Negative height - figure may be inverted!")
    return height_axis, pelvis_to_head[height_axis]

# Load RealWorld-1225 (raw)
rw_path = WORKSPACE_ROOT / 'datasets/real_world/real_data1225/val/label/frame_0.npy'
rw_raw = np.load(rw_path)
analyze_pose_orientation(rw_raw, "RealWorld-1225 (Original)")

# Apply coordinate transform
rw_transformed = PreprocessingUtils.coordinate_transform_y_to_z(rw_raw)
analyze_pose_orientation(rw_transformed, "RealWorld-1225 (After Y→Z Transform)")

# Load MMBody for comparison
mmbody_files = list((WORKSPACE_ROOT / 'datasets/mmBody/train').glob('*/mesh/frame_0.npz'))
if mmbody_files:
    with np.load(mmbody_files[0]) as data:
        mb = data['joints'][:22]
        analyze_pose_orientation(mb, "MMBody (Reference)")

print("\n" + "="*50)
print("✅ After transform, both datasets use Z-up coordinate system")

In [ ]:
# ============================================================
# Visual Comparison: Before and After Coordinate Transform
# ============================================================
# Compare RealWorld-1225 pose visualization before/after transform

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Define skeleton connections (22 joints SMPL-X format)
BONE_CONNECTIONS = [
    (0, 1), (0, 2), (0, 3),  # pelvis to legs and spine
    (1, 4), (2, 5), (3, 6),  # upper legs, lower spine
    (4, 7), (5, 8), (6, 9),  # knees, mid spine
    (7, 10), (8, 11), (9, 12), (9, 13), (9, 14),  # ankles, upper spine
    (12, 15), (13, 16), (14, 17),  # head, shoulders
    (16, 18), (17, 19),  # elbows
    (18, 20), (19, 21),  # wrists
]

BODY_PART_COLORS = {
    'spine': '#2E86AB',
    'left_leg': '#A23B72', 
    'right_leg': '#F18F01',
    'left_arm': '#C73E1D',
    'right_arm': '#3B1F2B'
}

BONE_PART_MAPPING = {
    (0, 1): 'left_leg', (1, 4): 'left_leg', (4, 7): 'left_leg', (7, 10): 'left_leg',
    (0, 2): 'right_leg', (2, 5): 'right_leg', (5, 8): 'right_leg', (8, 11): 'right_leg',
    (0, 3): 'spine', (3, 6): 'spine', (6, 9): 'spine', (9, 12): 'spine', (12, 15): 'spine',
    (9, 13): 'left_arm', (13, 16): 'left_arm', (16, 18): 'left_arm', (18, 20): 'left_arm',
    (9, 14): 'right_arm', (14, 17): 'right_arm', (17, 19): 'right_arm', (19, 21): 'right_arm',
}

def plot_pose_simple(joints, ax, title="Pose", show_axes=True):
    """Plot a single pose on 3D axis."""
    joints = np.array(joints)
    
    # Plot joints
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], 
              c='black', s=50, alpha=0.9, edgecolors='white', linewidth=1.5)
    
    # Plot bones
    for connection in BONE_CONNECTIONS:
        j1, j2 = connection
        if j1 >= len(joints) or j2 >= len(joints):
            continue
        body_part = BONE_PART_MAPPING.get(connection, 'spine')
        color = BODY_PART_COLORS[body_part]
        ax.plot([joints[j1, 0], joints[j2, 0]],
               [joints[j1, 1], joints[j2, 1]],
               [joints[j1, 2], joints[j2, 2]],
               color=color, linewidth=3.0, alpha=0.9)
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    
    # Equal aspect ratio
    max_range = np.ptp(joints, axis=0).max() / 2.0
    mid = joints.mean(axis=0)
    ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax.set_zlim(mid[2] - max_range, mid[2] + max_range)

# Create comparison figure
fig = plt.figure(figsize=(18, 6))

# 1. RealWorld-1225 Original (lying down due to wrong coord system)
ax1 = fig.add_subplot(131, projection='3d')
rw_norm_raw = PreprocessingUtils.pelvis_normalization(rw_raw)
plot_pose_simple(rw_norm_raw, ax1, "RealWorld-1225\n(Original - Wrong Coords)")
ax1.view_init(elev=20, azim=45)

# 2. RealWorld-1225 Transformed (standing up correctly)
ax2 = fig.add_subplot(132, projection='3d')
rw_norm_transformed = PreprocessingUtils.pelvis_normalization(rw_transformed)
plot_pose_simple(rw_norm_transformed, ax2, "RealWorld-1225\n(After Y→Z Transform)")
ax2.view_init(elev=20, azim=45)

# 3. MMBody Reference
ax3 = fig.add_subplot(133, projection='3d')
if mmbody_files:
    with np.load(mmbody_files[0]) as data:
        mb_norm = PreprocessingUtils.pelvis_normalization(data['joints'][:22])
        plot_pose_simple(mb_norm, ax3, "MMBody\n(Reference)")
        ax3.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig(WORKSPACE_ROOT / 'logs/coordinate_transform_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"📊 Saved comparison figure: {WORKSPACE_ROOT / 'logs/coordinate_transform_comparison.png'}")

In [ ]:
# ============================================================
# Load RealWorld-1225 Dataset with Automatic Transform
# ============================================================
# This demonstrates loading the dataset through the standard API,
# which automatically applies the coordinate transform.

import torch

# Load RealWorld-1225 using the dataset API (transform is automatic)
configs = [c for c in get_available_datasets(['wi-compass']) if c['name'] == 'RealWorld-1225']
if configs:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dataset, info = create_dataset(configs, num_joints=22, device=device, use_cache=False)
    
    # Get a sample
    sample, label = dataset[0]
    sample_np = sample.cpu().numpy()
    
    print(f"✅ Loaded RealWorld-1225 via API")
    print(f"   Sample shape: {sample_np.shape}")
    print(f"   Head position (should have positive Z): {sample_np[15]}")
    
    # Visualize
    fig = plt.figure(figsize=(12, 5))
    
    ax1 = fig.add_subplot(121, projection='3d')
    plot_pose_simple(sample_np, ax1, "RealWorld-1225 (Loaded via API)")
    ax1.view_init(elev=20, azim=45)
    
    ax2 = fig.add_subplot(122, projection='3d')
    plot_pose_simple(sample_np, ax2, "Side View")
    ax2.view_init(elev=10, azim=0)
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ RealWorld-1225 not found in config")

In [ ]:
def plot_single_pose(joints, ax, title="Human Pose", show_axes=True):
    """
    Plot a single pose on the specified 3D axis
    
    Args:
        joints: (num_joints, 3) numpy array
        ax: matplotlib 3D axis
        title: Figure title
        show_axes: Whether to show axes
    """
    joints = np.array(joints)
    
    # Set background style
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('lightgray')
    ax.yaxis.pane.set_edgecolor('lightgray')
    ax.zaxis.pane.set_edgecolor('lightgray')
    ax.xaxis.pane.set_alpha(0.1)
    ax.yaxis.pane.set_alpha(0.1)
    ax.zaxis.pane.set_alpha(0.1)
    
    # Decide whether to show axes based on show_axes parameter
    if not show_axes:
        # Hide axes
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_xlabel('')
        ax.set_ylabel('') 
        ax.set_zlabel('')
        
        # Hide axis lines and ticks
        ax.xaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        ax.yaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        ax.zaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        
        # Set grid to transparent
        ax.grid(False)
    else:
        # Show axes and labels
        ax.set_xlabel('X', fontsize=10)
        ax.set_ylabel('Y', fontsize=10) 
        ax.set_zlabel('Z', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Plot joint points
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], 
              c='black', s=50, alpha=0.9, edgecolors='white', linewidth=1.5)
    
    # Plot bone connections
    for connection in BONE_CONNECTIONS:
        joint1_idx, joint2_idx = connection
        
        if joint1_idx >= joints.shape[0] or joint2_idx >= joints.shape[0]:
            continue
            
        body_part = BONE_PART_MAPPING.get(connection, 'spine')
        color = BODY_PART_COLORS[body_part]
        
        x_coords = [joints[joint1_idx, 0], joints[joint2_idx, 0]]
        y_coords = [joints[joint1_idx, 1], joints[joint2_idx, 1]]
        z_coords = [joints[joint1_idx, 2], joints[joint2_idx, 2]]
        
        ax.plot(x_coords, y_coords, z_coords, 
               color=color, linewidth=4.0, alpha=0.9)
    
    # Set title
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    
    # Set equal axis aspect ratio
    max_range = np.array([joints[:, 0].max()-joints[:, 0].min(),
                         joints[:, 1].max()-joints[:, 1].min(),
                         joints[:, 2].max()-joints[:, 2].min()]).max() / 2.0
    
    mid_x = (joints[:, 0].max()+joints[:, 0].min()) * 0.5
    mid_y = (joints[:, 1].max()+joints[:, 1].min()) * 0.5
    mid_z = (joints[:, 2].max()+joints[:, 2].min()) * 0.5
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Set viewing angle
    ax.view_init(elev=10, azim=0)

In [ ]:
import numpy as np 
from pathlib import Path

# Update these paths to point to your local data
datapath_root = Path("../../../datasets/simulation_datasets/rfgen_data")
frame_id = 1442
data_path = datapath_root / f"output_pointclouds/sequence_1/radar/pointcloud_{frame_id}.npy"
groundtruth_path = Path("../../../logs/wicompass/sampled_poses/converted_poses.npy")


data = np.load(data_path)
label = np.load(groundtruth_path)[1,:,:]

In [ ]:
# Convert to numpy array
human_pose = np.array(label)
pointcloud = np.array(data)[:, :3]  # Only take XYZ coordinates, ignore intensity and other info

print(f"Pose data shape: {human_pose.shape}")
print(f"Point cloud data shape: {pointcloud.shape}")
print(f"Point cloud data range: X[{pointcloud[:, 0].min():.3f}, {pointcloud[:, 0].max():.3f}], Y[{pointcloud[:, 1].min():.3f}, {pointcloud[:, 1].max():.3f}], Z[{pointcloud[:, 2].min():.3f}, {pointcloud[:, 2].max():.3f}]")

# Create pose visualization function with point cloud
def plot_pose_with_pointcloud(joints, pointcloud, ax, title="Pose with Point Cloud", show_axes=True):
    """
    Plot pose and point cloud on the specified 3D axis
    
    Args:
        joints: (num_joints, 3) numpy array - Human joints
        pointcloud: (num_points, 3) numpy array - Point cloud data
        ax: matplotlib 3D axis
        title: Figure title
        show_axes: Whether to show axes
    """
    joints = np.array(joints)
    pointcloud = np.array(pointcloud)
    
    # Set background style
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('lightgray')
    ax.yaxis.pane.set_edgecolor('lightgray')
    ax.zaxis.pane.set_edgecolor('lightgray')
    ax.xaxis.pane.set_alpha(0.1)
    ax.yaxis.pane.set_alpha(0.1)
    ax.zaxis.pane.set_alpha(0.1)
    
    # Decide whether to show axes based on show_axes parameter
    if not show_axes:
        # Hide axes
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_xlabel('')
        ax.set_ylabel('') 
        ax.set_zlabel('')
        
        # Hide axis lines and ticks
        ax.xaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        ax.yaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        ax.zaxis.line.set_color((1.0, 1.0, 1.0, 0.0))
        
        # Set grid to transparent
        ax.grid(False)
    else:
        # Show axes and labels
        ax.set_xlabel('X', fontsize=10)
        ax.set_ylabel('Y', fontsize=10) 
        ax.set_zlabel('Z', fontsize=10)
        ax.grid(True, alpha=0.3)
    
    # Plot point cloud data (semi-transparent small points)
    if len(pointcloud) > 0:
        ax.scatter(pointcloud[:, 0], pointcloud[:, 1], pointcloud[:, 2], 
                  c='lightblue', s=10, alpha=0.3, label=f'Point Cloud ({len(pointcloud)} points)')
    
    # Plot joint points (larger red points, above point cloud)
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], 
              c='red', s=80, alpha=1.0, edgecolors='black', linewidth=2, 
              label='Joints', zorder=10)
    
    # Plot bone connections (on top layer)
    for connection in BONE_CONNECTIONS:
        joint1_idx, joint2_idx = connection
        
        if joint1_idx >= joints.shape[0] or joint2_idx >= joints.shape[0]:
            continue
            
        body_part = BONE_PART_MAPPING.get(connection, 'spine')
        color = BODY_PART_COLORS[body_part]
        
        x_coords = [joints[joint1_idx, 0], joints[joint2_idx, 0]]
        y_coords = [joints[joint1_idx, 1], joints[joint2_idx, 1]]
        z_coords = [joints[joint1_idx, 2], joints[joint2_idx, 2]]
        
        ax.plot(x_coords, y_coords, z_coords, 
               color=color, linewidth=4.0, alpha=0.9, zorder=5)
    
    # Set title
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    
    # Calculate appropriate display range (including pose and point cloud)
    all_data = np.vstack([joints, pointcloud]) if len(pointcloud) > 0 else joints
    
    max_range = np.array([all_data[:, 0].max()-all_data[:, 0].min(),
                         all_data[:, 1].max()-all_data[:, 1].min(),
                         all_data[:, 2].max()-all_data[:, 2].min()]).max() / 2.0
    
    mid_x = (all_data[:, 0].max()+all_data[:, 0].min()) * 0.5
    mid_y = (all_data[:, 1].max()+all_data[:, 1].min()) * 0.5
    mid_z = (all_data[:, 2].max()+all_data[:, 2].min()) * 0.5
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Set viewing angle
    ax.view_init(elev=10, azim=0)
    
    # Add legend
    ax.legend(loc='upper right', fontsize=8)

# Create figure and subplots
fig = plt.figure(figsize=(15, 8))

# Create three subplots with different views: pose + point cloud
ax1 = fig.add_subplot(131, projection='3d')
plot_pose_with_pointcloud(human_pose, pointcloud, ax1, "Front View - Pose + Point Cloud", show_axes=True)
ax1.view_init(elev=10, azim=0)  # Front view

ax2 = fig.add_subplot(132, projection='3d')
plot_pose_with_pointcloud(human_pose, pointcloud, ax2, "Side View - Pose + Point Cloud", show_axes=True)
ax2.view_init(elev=10, azim=90)  # Side view

ax3 = fig.add_subplot(133, projection='3d')
plot_pose_with_pointcloud(human_pose, pointcloud, ax3, "3D Perspective - Pose + Point Cloud", show_axes=True)
ax3.view_init(elev=20, azim=45)  # 3D perspective view

plt.tight_layout()
plt.show()

# Print statistics
print(f"\n=== Data Statistics ===")
print(f"Number of joints: {len(human_pose)}")
print(f"Number of point cloud points: {len(pointcloud)}")
print(f"\nJoint coordinates:")
for i, (joint_name, coord) in enumerate(zip(JOINT_NAMES, human_pose)):
    print(f"{i:2d}. {joint_name:15s}: [{coord[0]:8.3f}, {coord[1]:8.3f}, {coord[2]:8.3f}]")

In [ ]:
# Create detailed comparison visualization
fig = plt.figure(figsize=(18, 12))

# First row: separate visualizations of different data
# 1. Point cloud only
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
ax1.scatter(pointcloud[:, 0], pointcloud[:, 1], pointcloud[:, 2], 
           c='lightblue', s=2, alpha=0.6)
ax1.set_title('Point Cloud Only', fontsize=12, fontweight='bold')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.view_init(elev=10, azim=45)

# 2. Skeleton only
ax2 = fig.add_subplot(2, 3, 2, projection='3d')
plot_single_pose(human_pose, ax2, "Skeleton Only", show_axes=True)
ax2.view_init(elev=10, azim=45)

# 3. Pose + point cloud combined
ax3 = fig.add_subplot(2, 3, 3, projection='3d')
plot_pose_with_pointcloud(human_pose, pointcloud, ax3, "Combined View", show_axes=True)
ax3.view_init(elev=10, azim=45)

# Second row: combined visualization from different viewpoints
views = [
    {"elev": 10, "azim": 0, "title": "Front"},
    {"elev": 10, "azim": 90, "title": "Side"},
    {"elev": 90, "azim": 0, "title": "Top"}
]

for i, view in enumerate(views):
    ax = fig.add_subplot(2, 3, i+4, projection='3d')
    plot_pose_with_pointcloud(human_pose, pointcloud, ax, 
                             f"{view['title']} View", show_axes=False)
    ax.view_init(elev=view["elev"], azim=view["azim"])

plt.tight_layout()
plt.show()

# Analyze spatial relationship between point cloud and pose
print(f"\n=== Spatial Relationship Analysis ===")
print(f"Point cloud center: [{pointcloud.mean(axis=0)[0]:.3f}, {pointcloud.mean(axis=0)[1]:.3f}, {pointcloud.mean(axis=0)[2]:.3f}]")
print(f"Pose center: [{human_pose.mean(axis=0)[0]:.3f}, {human_pose.mean(axis=0)[1]:.3f}, {human_pose.mean(axis=0)[2]:.3f}]")

# Calculate minimum distance from each joint to point cloud
from scipy.spatial.distance import cdist
if len(pointcloud) > 0:
    distances = cdist(human_pose, pointcloud)
    min_distances = distances.min(axis=1)
    
    print(f"\n=== Minimum Distance from Joints to Point Cloud ===")
    for i, (joint_name, dist) in enumerate(zip(JOINT_NAMES, min_distances)):
        print(f"{i:2d}. {joint_name:15s}: {dist:.3f}m")

In [ ]:
# Create interactive multi-viewpoint visualization
def visualize_pose_interactive(joints, title="Human Pose"):
    """Create interactive pose visualization"""
    fig = plt.figure(figsize=(15, 10))
    
    # Define different viewpoints
    views = [
        {"elev": 10, "azim": 0, "title": "Front View"},
        {"elev": 10, "azim": 90, "title": "Side View"},
        {"elev": 10, "azim": 180, "title": "Back View"},
        {"elev": 10, "azim": 270, "title": "Side View (Right)"},
        {"elev": 90, "azim": 0, "title": "Top View"},
        {"elev": -10, "azim": 45, "title": "3D Perspective"}
    ]
    
    for i, view in enumerate(views):
        ax = fig.add_subplot(2, 3, i+1, projection='3d')
        plot_single_pose(joints, ax, f"{title} - {view['title']}", show_axes=False)
        ax.view_init(elev=view["elev"], azim=view["azim"])
    
    plt.tight_layout()
    plt.show()

# Visualize multiple viewpoints of current pose
visualize_pose_interactive(human_pose, "Human Pose Analysis")